# BioViL-T comparison on the same 50 ALBEF ITC cases

Loads the exact image IDs saved by the ALBEF ITC visualization notebook and compares both BioViL-T pathology maps in five columns:

1. Original CXR
2. Cardiomegaly heatmap
3. Cardiomegaly overlay
4. Pleural effusion heatmap
5. Pleural effusion overlay

The BioViL-T heatmaps remain aligned at `[16:240, 16:240]`; the excluded 16-pixel border is neither resized nor extrapolated.


In [ ]:
from pathlib import Path
import ast
import math
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from matplotlib import colormaps
from matplotlib.patches import Rectangle
from PIL import Image
from IPython.display import display

TARGET_LABELS = ['Cardiomegaly', 'Pleural effusion']
EXPECTED_IMAGES = 50
EXPECTED_IMAGE_RES = 256
CASES_PER_PAGE = 5
OVERLAY_ALPHA = 0.50
CMAP_NAME = 'magma'
FIG_DPI = 150
SAVE_OUTPUTS = True


In [ ]:
# EDIT THESE PATHS
# This is the CSV saved by the ALBEF ITC visualization notebook.
ALBEF_SELECTED_50_CSV = Path('/path/to/albef_visualization/selected_50_margin_cases.csv')

# This directory must contain manifest.csv and maps/ from the BioViL-T extraction
# run covering all VinDr test images for both target labels.
BIOVIL_OUTPUT_DIR = Path('/path/to/biovil_t_phrase_grounding_all_images')
IMAGES_ROOT = Path('/path/to/vindr_256_pngs')
ANNOTATIONS_CSV = Path('/path/to/vindr_annotations.csv')
IMAGE_METADATA_CSV = Path('/path/to/vindr_image_metadata.csv')
VIS_OUTPUT_DIR = Path('/path/to/biovil_t_same_50_comparison')

MANIFEST_CSV = BIOVIL_OUTPUT_DIR / 'manifest.csv'
for path in [ALBEF_SELECTED_50_CSV, MANIFEST_CSV, IMAGES_ROOT,
             ANNOTATIONS_CSV, IMAGE_METADATA_CSV]:
    if not path.exists():
        raise FileNotFoundError(path)
VIS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('ALBEF selection:', ALBEF_SELECTED_50_CSV)
print('BioViL manifest:', MANIFEST_CSV)
print('Outputs:', VIS_OUTPUT_DIR)


## Load the exact ALBEF selection and match both BioViL-T maps

In [ ]:
def choose_column(df, candidates, purpose):
    lookup = {str(c).casefold(): c for c in df.columns}
    for candidate in candidates:
        if candidate.casefold() in lookup:
            return lookup[candidate.casefold()]
    raise KeyError(f'Cannot find {purpose}. Available columns: {list(df.columns)}')

selected_df = pd.read_csv(ALBEF_SELECTED_50_CSV)
selected_id_col = choose_column(selected_df, ['image_id', 'imageid'], 'selected image ID')
selected_df = selected_df.rename(columns={selected_id_col: 'image_id'})
selected_df['image_id'] = selected_df['image_id'].astype(str)

if len(selected_df) != EXPECTED_IMAGES:
    raise ValueError(f'Expected exactly 50 selected rows, found {len(selected_df)}')
if selected_df['image_id'].duplicated().any():
    raise ValueError('ALBEF selection contains duplicate image IDs')

manifest_df = pd.read_csv(MANIFEST_CSV)
required = {'image_id', 'label', 'heatmap_path'}
missing = required - set(manifest_df.columns)
if missing:
    raise KeyError(f'Manifest is missing columns: {sorted(missing)}')
manifest_df['image_id'] = manifest_df['image_id'].astype(str)
manifest_df = manifest_df[manifest_df['label'].isin(TARGET_LABELS)].copy()

def resolve_heatmap_path(saved_path):
    saved_path = Path(str(saved_path))
    if saved_path.is_file():
        return saved_path
    return BIOVIL_OUTPUT_DIR / 'maps' / saved_path.name

manifest_df['heatmap_path'] = manifest_df['heatmap_path'].map(resolve_heatmap_path)
if manifest_df.duplicated(['image_id', 'label']).any():
    dupes = manifest_df[manifest_df.duplicated(['image_id', 'label'], keep=False)]
    raise ValueError(f'Duplicate BioViL-T image-label pairs:\n{dupes[["image_id", "label"]]}')

selected_ids = set(selected_df['image_id'])
matched = manifest_df[manifest_df['image_id'].isin(selected_ids)].copy()
pair_counts = matched.groupby('image_id')['label'].nunique()
bad_ids = sorted(selected_ids - set(pair_counts[pair_counts == 2].index))
if bad_ids:
    raise ValueError(f'Missing Cardiomegaly and/or Pleural-effusion maps for {len(bad_ids)} selected images: {bad_ids[:10]}')

path_lookup = matched.set_index(['image_id', 'label'])['heatmap_path']
selected_df['image_path'] = selected_df['image_id'].map(lambda x: IMAGES_ROOT / f'{x}.png')
selected_df['cardiomegaly_heatmap_path'] = selected_df['image_id'].map(
    lambda x: path_lookup.loc[(x, 'Cardiomegaly')])
selected_df['pleural_effusion_heatmap_path'] = selected_df['image_id'].map(
    lambda x: path_lookup.loc[(x, 'Pleural effusion')])

missing_files = []
for row in selected_df.itertuples(index=False):
    for path in [row.image_path, row.cardiomegaly_heatmap_path,
                 row.pleural_effusion_heatmap_path]:
        if not Path(path).is_file():
            missing_files.append(str(path))
if missing_files:
    raise FileNotFoundError(f'{len(missing_files)} required files are missing. First entries: {missing_files[:10]}')

print(f'Matched both pathology maps for all {len(selected_df)} ALBEF-selected images.')
display(selected_df[['image_id', 'cardiomegaly_heatmap_path',
                     'pleural_effusion_heatmap_path']].head())


## Load ground-truth annotations and original dimensions

In [ ]:
ann_raw = pd.read_csv(ANNOTATIONS_CSV)
ann_map = {
    'image_id': choose_column(ann_raw, ['image_id', 'imageid'], 'annotation image ID'),
    'class_name': choose_column(ann_raw, ['class_name', 'class', 'label'], 'annotation class'),
    'x_min': choose_column(ann_raw, ['x_min', 'xmin', 'x1'], 'x_min'),
    'y_min': choose_column(ann_raw, ['y_min', 'ymin', 'y1'], 'y_min'),
    'x_max': choose_column(ann_raw, ['x_max', 'xmax', 'x2'], 'x_max'),
    'y_max': choose_column(ann_raw, ['y_max', 'ymax', 'y2'], 'y_max'),
}
annotations_df = ann_raw[list(ann_map.values())].rename(columns={v: k for k, v in ann_map.items()})
annotations_df['image_id'] = annotations_df['image_id'].astype(str)

meta_raw = pd.read_csv(IMAGE_METADATA_CSV)
meta_map = {
    'image_id': choose_column(meta_raw, ['image_id', 'imageid'], 'metadata image ID'),
    'width': choose_column(meta_raw, ['width', 'image_width', 'original_width', 'w'], 'original width'),
    'height': choose_column(meta_raw, ['height', 'image_height', 'original_height', 'h'], 'original height'),
}
metadata_df = meta_raw[list(meta_map.values())].rename(columns={v: k for k, v in meta_map.items()})
metadata_df['image_id'] = metadata_df['image_id'].astype(str)
if metadata_df['image_id'].duplicated().any():
    raise ValueError('Image metadata contains duplicate image IDs')
metadata_lookup = metadata_df.set_index('image_id')[['width', 'height']].to_dict('index')
print(f'Annotations={len(annotations_df):,}; metadata={len(metadata_df):,}')


## Validate the 100 BioViL-T maps used in this comparison

In [ ]:
def safe_torch_load(path):
    try:
        return torch.load(path, map_location='cpu', weights_only=False)
    except TypeError:
        return torch.load(path, map_location='cpu')

def as_2d_numpy(value):
    if torch.is_tensor(value):
        value = value.detach().cpu().float().numpy()
    return np.asarray(value, dtype=np.float32).squeeze()

def parse_valid_region(item):
    region = item.get('valid_region', [16, 240, 16, 240])
    if isinstance(region, str):
        region = ast.literal_eval(region)
    if len(region) != 4:
        raise ValueError(f'Invalid valid_region: {region}')
    y0, y1, x0, x1 = map(int, region)
    if not (0 <= y0 < y1 <= EXPECTED_IMAGE_RES and 0 <= x0 < x1 <= EXPECTED_IMAGE_RES):
        raise ValueError(f'Out-of-range valid_region: {region}')
    return y0, y1, x0, x1

def validate_heatmap(path, image_id, expected_label):
    item = safe_torch_load(path)
    required = ['image_id', 'label', 'similarity_map_raw', 'similarity_map_vis']
    missing = [key for key in required if key not in item]
    if missing:
        raise KeyError(f'{path}: missing payload keys {missing}')
    if str(item['image_id']) != str(image_id) or str(item['label']) != expected_label:
        raise ValueError(f'{path}: payload identity mismatch')
    raw = as_2d_numpy(item['similarity_map_raw'])
    vis = as_2d_numpy(item['similarity_map_vis'])
    if raw.shape != (256, 256) or vis.shape != (256, 256):
        raise ValueError(f'{path}: expected 256x256 maps, got raw={raw.shape}, vis={vis.shape}')
    if not np.isfinite(raw).all() or not np.isfinite(vis).all():
        raise ValueError(f'{path}: map contains NaN or infinity')
    region = parse_valid_region(item)
    y0, y1, x0, x1 = region
    outside = np.ones((256, 256), dtype=bool)
    outside[y0:y1, x0:x1] = False
    if not np.allclose(raw[outside], 0, atol=1e-7) or not np.allclose(vis[outside], 0, atol=1e-7):
        raise ValueError(f'{path}: nonzero values outside valid_region')
    central_vis = vis[y0:y1, x0:x1]
    if central_vis.min() < -1e-6 or central_vis.max() > 1 + 1e-6:
        raise ValueError(f'{path}: similarity_map_vis is outside [0,1]')
    central_raw = raw[y0:y1, x0:x1]
    stats = {'raw_min': float(central_raw.min()), 'raw_max': float(central_raw.max()),
             'raw_mean': float(central_raw.mean()), 'raw_std': float(central_raw.std())}
    return item, vis, region, stats

records = []
for row in selected_df.itertuples(index=False):
    record = {'image_id': row.image_id}
    for label, path in [('Cardiomegaly', row.cardiomegaly_heatmap_path),
                        ('Pleural effusion', row.pleural_effusion_heatmap_path)]:
        item, vis, region, stats = validate_heatmap(path, row.image_id, label)
        slug = label.lower().replace(' ', '_')
        record.update({f'{slug}_{key}': value for key, value in stats.items()})
        record[f'{slug}_prompt'] = item.get('prompt', label)
        record[f'{slug}_ground_truth'] = int(item.get('ground_truth', -1))
    records.append(record)

diagnostics_df = pd.DataFrame(records)
print(f'All {2 * len(selected_df)} maps passed validation.')
display(diagnostics_df.head())


## Image, overlay, and ground-truth box helpers

In [ ]:
def load_image(image_id):
    with Image.open(IMAGES_ROOT / f'{image_id}.png') as handle:
        return handle.convert('RGB')

def make_overlay(image, vis_map, valid_region, alpha=OVERLAY_ALPHA):
    rgb = np.asarray(image, dtype=np.float32) / 255.0
    heatmap = as_2d_numpy(vis_map)
    if heatmap.shape != rgb.shape[:2]:
        raise ValueError(f'Image/map mismatch: image={rgb.shape[:2]}, map={heatmap.shape}')
    y0, y1, x0, x1 = valid_region
    valid_mask = np.zeros_like(heatmap, dtype=bool)
    valid_mask[y0:y1, x0:x1] = True
    color = colormaps[CMAP_NAME](np.clip(heatmap, 0, 1))[..., :3]
    alpha_map = (alpha * np.clip(heatmap, 0, 1) * valid_mask)[..., None]
    return np.clip((1 - alpha_map) * rgb + alpha_map * color, 0, 1)

def get_boxes(image_id, label, displayed_size):
    subset = annotations_df[(annotations_df.image_id == str(image_id)) &
                            (annotations_df.class_name == label)]
    if subset.empty:
        return []
    if str(image_id) not in metadata_lookup:
        raise KeyError(f'Missing original dimensions for {image_id}')
    original = metadata_lookup[str(image_id)]
    sx = displayed_size[0] / float(original['width'])
    sy = displayed_size[1] / float(original['height'])
    boxes = []
    for row in subset.itertuples(index=False):
        x1 = np.clip(float(row.x_min) * sx, 0, displayed_size[0])
        y1 = np.clip(float(row.y_min) * sy, 0, displayed_size[1])
        x2 = np.clip(float(row.x_max) * sx, 0, displayed_size[0])
        y2 = np.clip(float(row.y_max) * sy, 0, displayed_size[1])
        if x2 > x1 and y2 > y1:
            boxes.append((x1, y1, x2, y2))
    return boxes

def draw_boxes(ax, boxes, color):
    for x1, y1, x2, y2 in boxes:
        ax.add_patch(Rectangle((x1, y1), x2-x1, y2-y1,
                               fill=False, edgecolor=color, linewidth=2))


## Visualize the same 50 images in five columns

In [ ]:
def create_gallery_page(page_df, page_number, start_index):
    fig, axes = plt.subplots(len(page_df), 5, figsize=(18, 3.5 * len(page_df)),
                             dpi=FIG_DPI, squeeze=False)
    for r, row in enumerate(page_df.itertuples(index=False)):
        image = load_image(row.image_id)
        cardio_item, cardio_vis, cardio_region, cardio_stats = validate_heatmap(
            row.cardiomegaly_heatmap_path, row.image_id, 'Cardiomegaly')
        eff_item, eff_vis, eff_region, eff_stats = validate_heatmap(
            row.pleural_effusion_heatmap_path, row.image_id, 'Pleural effusion')

        cardio_boxes = get_boxes(row.image_id, 'Cardiomegaly', image.size)
        eff_boxes = get_boxes(row.image_id, 'Pleural effusion', image.size)
        cardio_gt = int(cardio_item.get('ground_truth', -1))
        eff_gt = int(eff_item.get('ground_truth', -1))

        axes[r, 0].imshow(image)
        draw_boxes(axes[r, 0], cardio_boxes, 'lime')
        draw_boxes(axes[r, 0], eff_boxes, 'cyan')
        axes[r, 0].set_title(
            f'{start_index+r+1:02d}. {row.image_id}\nCardio GT={cardio_gt} | Effusion GT={eff_gt}\n'
            'lime=Cardio | cyan=Effusion', fontsize=8)

        axes[r, 1].imshow(cardio_vis, cmap=CMAP_NAME, vmin=0, vmax=1)
        axes[r, 1].set_title(
            f'Cardiomegaly heatmap\nraw min={cardio_stats["raw_min"]:.4f} | '
            f'max={cardio_stats["raw_max"]:.4f}\nmean={cardio_stats["raw_mean"]:.4f}', fontsize=8)

        axes[r, 2].imshow(make_overlay(image, cardio_vis, cardio_region))
        draw_boxes(axes[r, 2], cardio_boxes, 'lime')
        axes[r, 2].set_title('Cardiomegaly overlay', fontsize=8)

        axes[r, 3].imshow(eff_vis, cmap=CMAP_NAME, vmin=0, vmax=1)
        axes[r, 3].set_title(
            f'Pleural effusion heatmap\nraw min={eff_stats["raw_min"]:.4f} | '
            f'max={eff_stats["raw_max"]:.4f}\nmean={eff_stats["raw_mean"]:.4f}', fontsize=8)

        axes[r, 4].imshow(make_overlay(image, eff_vis, eff_region))
        draw_boxes(axes[r, 4], eff_boxes, 'cyan')
        axes[r, 4].set_title('Pleural effusion overlay', fontsize=8)

        for ax in axes[r]:
            ax.axis('off')

    fig.suptitle(f'BioViL-T — same 50 ALBEF ITC cases — page {page_number:02d}',
                 fontsize=14, fontweight='bold', y=1.002)
    plt.tight_layout()
    return fig

num_pages = math.ceil(len(selected_df) / CASES_PER_PAGE)
print(f'Rendering {len(selected_df)} images across {num_pages} pages...')
for page_index in range(num_pages):
    start = page_index * CASES_PER_PAGE
    page = selected_df.iloc[start:start + CASES_PER_PAGE]
    fig = create_gallery_page(page, page_index + 1, start)
    if SAVE_OUTPUTS:
        output_path = VIS_OUTPUT_DIR / f'biovil_t_same_50_page_{page_index+1:02d}.png'
        fig.savefig(output_path, dpi=FIG_DPI, bbox_inches='tight')
        print('Saved:', output_path)
    plt.show()
    plt.close(fig)


## Save the exact comparison order and diagnostics

In [ ]:
order_output = selected_df[['image_id', 'image_path',
                                    'cardiomegaly_heatmap_path',
                                    'pleural_effusion_heatmap_path']].merge(
    diagnostics_df, on='image_id', how='left', validate='one_to_one')
order_path = VIS_OUTPUT_DIR / 'same_50_albef_itc_biovil_t_comparison_order.csv'
order_output.to_csv(order_path, index=False)
print('Saved:', order_path)
display(order_output.head(10))


## Interpretation notes

- Every row is one of the exact 50 image IDs from the ALBEF selection CSV.
- Both pathology maps are shown for each image, producing 100 BioViL-T heatmaps in total.
- Heatmap color is restricted to the valid `[16:240, 16:240]` region; the outer border is not colored in overlays.
- Ground-truth boxes are lime for Cardiomegaly and cyan for Pleural effusion.
- `similarity_map_vis` is normalized independently per map. Use the displayed raw statistics when judging whether apparent intensity reflects a strong or weak raw similarity map.
